In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_log_error

In [38]:
class BoxOfficeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(
            y.values if isinstance(y, pd.Series) else y,
            dtype=torch.float32,
        )

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
def rmsle_loss(y_pred, y_true):
    y_pred = torch.clamp(y_pred, min=0.0)
    y_true = torch.clamp(y_true, min=0.0)
    return torch.sqrt(torch.mean((torch.log1p(y_pred) - torch.log1p(y_true)) ** 2))

In [39]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.act(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        out = out + residual
        out = self.act(out)
        return out


class ResMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.2):
        super().__init__()
        self.input_fc = nn.Linear(input_dim, hidden_dim)
        self.input_bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [ResidualBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)]
        )
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.input_fc(x)
        x = self.input_bn(x)
        x = self.act(x)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.out(x).squeeze(-1)
        return x

In [40]:
def train_epoch(model, loader, optimizer, device, criterion, l2_weight=0.0):
    model.train()
    total_loss = 0.0
    n = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        if l2_weight > 0.0:
            l2 = 0.0
            for p in model.parameters():
                l2 = l2 + torch.sum(p ** 2)
            loss = loss + l2_weight * l2

        loss.backward()
        optimizer.step()

        bs = y_batch.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / n


@torch.no_grad()
def eval_epoch(model, loader, device, criterion):
    model.eval()
    total_loss = 0.0
    n = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        bs = y_batch.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / n


In [41]:
def run_experiment(config, X_train_scaled, X_val_scaled, y_train, y_val, device):
    train_dataset = BoxOfficeDataset(X_train_scaled, y_train)
    val_dataset = BoxOfficeDataset(X_val_scaled, y_val)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    model = ResMLP(
        input_dim=X_train_scaled.shape[1],
        hidden_dim=256,
        num_blocks=3,
        dropout=config.get("dropout", 0.0),
    ).to(device)

    if config.get("loss", "mse") == "rmsle":
        criterion = rmsle_loss
    else:
        criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get("lr", 1e-3),
        weight_decay=config.get("weight_decay", 0.0),
    )

    best_val = float("inf")
    best_epoch = 0
    patience = config.get("patience", 0)
    patience_counter = 0
    max_epochs = config.get("max_epochs", 50)
    use_early_stopping = config.get("early_stopping", False)

    for epoch in range(1, max_epochs + 1):
        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            criterion,
            l2_weight=config.get("l2_weight", 0.0),
        )
        val_loss = eval_epoch(model, val_loader, device, criterion)

        print(
            f"[{config['name']}] Epoch {epoch:03d} | "
            f"Train: {train_loss:.4f} | Val: {val_loss:.4f}"
        )

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_epoch = epoch
            patience_counter = 0
        else:
            if use_early_stopping:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"[{config['name']}] Early stopping at epoch {epoch}")
                    break

    return best_val, best_epoch


In [42]:
data = pd.read_csv(r"tmdb-box-office-data/train_processed.csv")
y = data["revenue_log"]
X = data.drop(columns=["revenue_log", "id", "revenue"], errors="ignore")
X = X.select_dtypes(include=["number", "bool"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

device = torch.device("cpu")


In [43]:
configs = [
    {
        "name": "mse_baseline_no_reg",
        "dropout": 0.0,
        "weight_decay": 0.0,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "mse",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "mse_dropout_only",
        "dropout": 0.2,
        "weight_decay": 0.0,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "mse",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "mse_l2_only",
        "dropout": 0.0,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "mse",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "mse_dropout_plus_l2",
        "dropout": 0.2,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "mse",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "mse_dropout_l2_earlystop",
        "dropout": 0.2,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": True,
        "patience": 5,
        "loss": "mse",
        "max_epochs": 100,
        "lr": 1e-3,
    },

    {
        "name": "rmsle_base",
        "dropout": 0.0,
        "weight_decay": 0.0,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "rmsle",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "rmsle_l2",
        "dropout": 0.0,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "rmsle",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "rmsle_dropout",
        "dropout": 0.2,
        "weight_decay": 0.0,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "rmsle",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "rmsle_dropout_l2",
        "dropout": 0.2,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": False,
        "patience": 0,
        "loss": "rmsle",
        "max_epochs": 50,
        "lr": 1e-3,
    },
    {
        "name": "rmsle_dropout_l2_earlystop",
        "dropout": 0.2,
        "weight_decay": 1e-5,
        "l2_weight": 0.0,
        "early_stopping": True,
        "patience": 5,
        "loss": "rmsle",
        "max_epochs": 100,
        "lr": 1e-3,
    },
]


In [44]:
results = []
for cfg in configs:
    print("=" * 80)
    best_val, best_epoch = run_experiment(
        cfg,
        X_train_scaled,
        X_val_scaled,
        y_train,
        y_val,
        device=device,
    )
    results.append(
        {
            "name": cfg["name"],
            "best_val_loss": best_val,
            "best_epoch": best_epoch,
            "loss_type": cfg["loss"],
            "dropout": cfg["dropout"],
            "weight_decay": cfg["weight_decay"],
            "early_stopping": cfg["early_stopping"],
        }
    )

results_df = pd.DataFrame(results)
print("\n=== Summary (lower is better) ===")
print(results_df.sort_values("best_val_loss"))

[mse_baseline_no_reg] Epoch 001 | Train: 110.9244 | Val: 25.0048
[mse_baseline_no_reg] Epoch 002 | Train: 6.4966 | Val: 6.1390
[mse_baseline_no_reg] Epoch 003 | Train: 4.6000 | Val: 5.5553
[mse_baseline_no_reg] Epoch 004 | Train: 3.7688 | Val: 5.4595
[mse_baseline_no_reg] Epoch 005 | Train: 3.5800 | Val: 6.4711
[mse_baseline_no_reg] Epoch 006 | Train: 2.8308 | Val: 6.0985
[mse_baseline_no_reg] Epoch 007 | Train: 2.6439 | Val: 6.0876
[mse_baseline_no_reg] Epoch 008 | Train: 2.0520 | Val: 6.0973
[mse_baseline_no_reg] Epoch 009 | Train: 1.6955 | Val: 6.5532
[mse_baseline_no_reg] Epoch 010 | Train: 1.7972 | Val: 5.7397
[mse_baseline_no_reg] Epoch 011 | Train: 1.5180 | Val: 5.6675
[mse_baseline_no_reg] Epoch 012 | Train: 1.3123 | Val: 5.0991
[mse_baseline_no_reg] Epoch 013 | Train: 1.1072 | Val: 5.7348
[mse_baseline_no_reg] Epoch 014 | Train: 0.8623 | Val: 5.6256
[mse_baseline_no_reg] Epoch 015 | Train: 0.7476 | Val: 5.1551
[mse_baseline_no_reg] Epoch 016 | Train: 0.8665 | Val: 5.3991
[mse_